# Bearing RUL Prediction Pipeline
This notebook predicts the Remaining Useful Life (RUL) of ball bearings 
using vibration sensor data from the PRONOSTIA/FEMTO-ST dataset.

## Step 1: Explore Data Structure
We first inspect one bearing folder to understand the file format.
Each bearing has hundreds of CSV files — one per 10-second recording window.
Each file contains 2560 vibration samples from 2 accelerometers (X and Y axis).

In [ ]:
import os
import pandas as pd

# Check what's inside one learning bearing
path = "Learning_set/Bearing1_1"
files = sorted(os.listdir(path))
print(f"Number of files: {len(files)}")
print(f"First 3 files: {files[:3]}")
print(f"Last 3 files: {files[-3:]}")

# Peek at one file
df = pd.read_csv(f"{path}/{files[0]}", header=None)
print(f"\nShape of one file: {df.shape}")
print(df.head())

## Step 2: Feature Extraction + Label Construction
We extract statistical features from each 10-second window:
- RMS: overall vibration energy
- Peak: maximum vibration amplitude
- Kurtosis: detects impulse-type faults (spikes)
- Skewness: signal asymmetry
- Std: signal spread
- Crest Factor: ratio of peak to RMS, sensitive to early faults

RUL label = total_lifetime - current_time_step
This is done for all 6 learning bearings (full lifetime known).

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew
from numpy.fft import fft

def extract_features(filepath):
    df_raw = pd.read_csv(filepath, header=None)
    ax = df_raw[4].values
    ay = df_raw[5].values if 5 in df_raw.columns else ax
    features = {}
    for name, sig in [("x", ax), ("y", ay)]:
        # Time domain
        features[f"rms_{name}"]      = np.sqrt(np.mean(sig**2))
        features[f"peak_{name}"]     = np.max(np.abs(sig))
        features[f"kurtosis_{name}"] = kurtosis(sig)
        features[f"skew_{name}"]     = skew(sig)
        features[f"std_{name}"]      = np.std(sig)
        features[f"crest_{name}"]    = np.max(np.abs(sig)) / (np.sqrt(np.mean(sig**2)) + 1e-10)
        # Frequency domain (FFT)
        fft_vals = np.abs(fft(sig))[:len(sig)//2]
        freqs    = np.linspace(0, 12800, len(fft_vals))
        features[f"fft_mean_{name}"]     = np.mean(fft_vals)
        features[f"fft_std_{name}"]      = np.std(fft_vals)
        features[f"fft_peak_{name}"]     = np.max(fft_vals)
        features[f"fft_kurtosis_{name}"] = kurtosis(fft_vals)
        features[f"spectral_entropy_{name}"] = -np.sum(
            (fft_vals/np.sum(fft_vals)) * np.log(fft_vals/np.sum(fft_vals) + 1e-10)
        )
        features[f"energy_low_{name}"]  = np.sum(fft_vals[freqs < 1000]**2)
        features[f"energy_mid_{name}"]  = np.sum(fft_vals[(freqs >= 1000) & (freqs < 5000)]**2)
        features[f"energy_high_{name}"] = np.sum(fft_vals[freqs >= 5000]**2)
    return features

def load_bearing(path):
    files = sorted(os.listdir(path))
    records = []
    for i, f in enumerate(files):
        feats = extract_features(os.path.join(path, f))
        feats["time_step"]   = i
        feats["total_steps"] = len(files)
        feats["RUL"]         = len(files) - i - 1
        records.append(feats)
    return pd.DataFrame(records)

# Load all 6 learning bearings
learning_bearings = [
    "Learning_set/Bearing1_1", "Learning_set/Bearing1_2",
    "Learning_set/Bearing2_1", "Learning_set/Bearing2_2",
    "Learning_set/Bearing3_1", "Learning_set/Bearing3_2",
]

all_data = []
for path in learning_bearings:
    df = load_bearing(path)
    df["bearing"] = path.split("/")[-1]
    all_data.append(df)
    print(f"  {path.split('/')[-1]}: done")

train_df = pd.concat(all_data, ignore_index=True)
print(f"\nShape: {train_df.shape}")
print(train_df.head(3))

## Step 3: Save Processed Data
We save the extracted features and RUL labels to a CSV file
so teammates can load it directly without reprocessing raw files.

In [ ]:
train_df.to_csv("train_features_v2.csv", index=False)
print("Saved!")